<a href="https://colab.research.google.com/github/saatour/GW170817/blob/main/GW190814_WEAVE_FITS_builder_20092026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from pathlib import Path
from io import StringIO
import json
import urllib.parse
import urllib.request

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Settings and original target list
# ------------------------------------------------------------

INPUT_FILE = Path("galaxy.csv")
OUTPUT_DIR = Path("GW190814_target_review")
OUTPUT_DIR.mkdir(exist_ok=True)

targets = pd.read_csv(
    INPUT_FILE,
    dtype={"coadd_object_id": "string"}
)

required = {"coadd_object_id", "ra", "dec"}
if not required.issubset(targets.columns):
    raise ValueError(f"galaxy.csv must contain: {sorted(required)}")

targets["coadd_object_id"] = targets["coadd_object_id"].str.strip()

if targets["coadd_object_id"].isna().any():
    raise ValueError("Missing target IDs.")

if not targets["coadd_object_id"].str.fullmatch(r"\d+").all():
    raise ValueError("Target IDs must contain digits only.")

if len(targets) != 273 or targets["coadd_object_id"].duplicated().any():
    raise ValueError("Expected 273 targets with unique IDs.")

print(f"Loaded {len(targets)} original targets.")


# ------------------------------------------------------------
# 2. Query DES DR2 and its Gaia DR3 crossmatch
#
# No magnitude or morphology cuts are applied here.
# IDs are read as strings to preserve every digit.
# ------------------------------------------------------------

ids_sql = ",".join(targets["coadd_object_id"])

sql = f"""
SELECT
    d.coadd_object_id,
    d.alphawin_j2000,
    d.deltawin_j2000,
    d.mag_auto_g,
    d.magerr_auto_g,
    d.mag_auto_r,
    d.magerr_auto_r,
    d.mag_auto_i,
    d.magerr_auto_i,
    d.mag_auto_g_dered,
    d.mag_auto_r_dered,
    d.mag_auto_i_dered,
    d.extended_class_coadd,
    d.spread_model_i,
    d.flags_g,
    d.flags_r,
    d.flags_i,
    d.imaflags_iso_g,
    d.imaflags_iso_r,
    d.imaflags_iso_i,
    x.id2 AS gaia_source_id,
    g.ra AS gaia_ra,
    g.dec AS gaia_dec,
    g.parallax,
    g.parallax_error,
    g.pmra,
    g.pmra_error,
    g.pmdec,
    g.pmdec_error
FROM des_dr2.main AS d
LEFT JOIN des_dr2.x1p5__main__gaia_dr3__gaia_source AS x
    ON d.coadd_object_id = x.id1
LEFT JOIN gaia_dr3.gaia_source AS g
    ON x.id2 = g.source_id
WHERE d.coadd_object_id IN ({ids_sql})
"""

params = {
    "sql": sql,
    "ofmt": "csv",
    "async": "False",
    "out": "",
    "profile": "default",
}

url = (
    "https://datalab.noirlab.edu/query/query?"
    + urllib.parse.urlencode(params)
)

request = urllib.request.Request(
    url,
    headers={"X-DL-AuthToken": "anonymous.0.0.anon_access"},
)

print("Querying DES DR2 and Gaia DR3...")

with urllib.request.urlopen(request, timeout=120) as response:
    csv_text = response.read().decode("utf-8")

# Read everything as text initially: Gaia IDs must never become floats.
raw = pd.read_csv(StringIO(csv_text), dtype="string")

if "coadd_object_id" not in raw.columns:
    raise RuntimeError("Unexpected query response:\n" + csv_text[:500])

if set(raw["coadd_object_id"]) != set(targets["coadd_object_id"]):
    missing = set(targets["coadd_object_id"]) - set(raw["coadd_object_id"])
    raise RuntimeError(f"DES target-ID mismatch. Missing IDs: {missing}")

# Save the exact response before processing.
(OUTPUT_DIR / "DES_Gaia_raw.csv").write_text(csv_text)
(OUTPUT_DIR / "DES_query.sql").write_text(sql)


# ------------------------------------------------------------
# 3. Helpers for quality checks
# ------------------------------------------------------------

def numeric(value):
    """Return a finite number, or None for missing/invalid values."""
    try:
        result = float(value)
        return result if np.isfinite(result) else None
    except (TypeError, ValueError):
        return None


def separation_arcsec(ra1, dec1, ra2, dec2):
    """Great-circle angular separation."""
    a, b, c, d = np.radians([ra1, dec1, ra2, dec2])
    h = (
        np.sin((d - b) / 2) ** 2
        + np.cos(b) * np.cos(d) * np.sin((c - a) / 2) ** 2
    )
    return float(
        np.degrees(2 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))) * 3600
    )


# ------------------------------------------------------------
# 4. Build one review row per original target
#
# A target may have multiple Gaia matches. Preserve those
# candidates without duplicating the galaxy's review row.
# ------------------------------------------------------------

groups = {
    object_id: group
    for object_id, group in raw.groupby("coadd_object_id", sort=False)
}

review_rows = []

for _, original in targets.iterrows():

    object_id = original["coadd_object_id"]
    matches = groups[object_id]
    des = matches.iloc[0]

    ra = numeric(des["alphawin_j2000"])
    dec = numeric(des["deltawin_j2000"])

    if ra is None or dec is None:
        raise ValueError(f"Missing DES coordinates for {object_id}")

    flags = []

    coordinate_difference = separation_arcsec(
        float(original["ra"]), float(original["dec"]), ra, dec
    )

    if coordinate_difference > 0.3:
        flags.append("coordinate_difference_gt_0.3arcsec")

    morphology = numeric(des["extended_class_coadd"])

    if morphology in (0, 1):
        flags.append("stellar_morphology_candidate")
    elif morphology != 3:
        flags.append("morphology_review")

    for band in ("g", "r", "i"):

        magnitude = numeric(des[f"mag_auto_{band}"])
        error = numeric(des[f"magerr_auto_{band}"])

        if (
            magnitude is None
            or not -10 < magnitude < 90
            or error is None
            or not 0 < error < 90
        ):
            flags.append(f"missing_or_invalid_{band}_photometry")

        extraction_flag = numeric(des[f"flags_{band}"])
        image_flag = numeric(des[f"imaflags_iso_{band}"])

        if extraction_flag is None:
            flags.append(f"missing_extraction_flag_{band}")
        elif extraction_flag != 0:
            flags.append(f"extraction_flag_{band}")

        if image_flag is None:
            flags.append(f"missing_image_flag_{band}")
        elif image_flag != 0:
            flags.append(f"image_mask_flag_{band}")

    gaia_matches = matches[
        matches["gaia_source_id"].notna()
        & matches["gaia_source_id"].str.strip().ne("")
    ]

    if len(gaia_matches) > 1:
        flags.append("multiple_gaia_matches")

    gaia_diagnostics = []

    for _, gaia in gaia_matches.iterrows():

        parallax = numeric(gaia["parallax"])
        parallax_error = numeric(gaia["parallax_error"])
        pmra = numeric(gaia["pmra"])
        pmra_error = numeric(gaia["pmra_error"])
        pmdec = numeric(gaia["pmdec"])
        pmdec_error = numeric(gaia["pmdec_error"])

        parallax_snr = None
        pm_snr = None

        if (
            parallax is not None
            and parallax_error is not None
            and parallax_error > 0
        ):
            parallax_snr = parallax / parallax_error

        if (
            all(
                x is not None
                for x in (pmra, pmra_error, pmdec, pmdec_error)
            )
            and pmra_error > 0
            and pmdec_error > 0
        ):
            # Screening proxy only: does not include covariance.
            pm_snr = float(
                np.hypot(pmra, pmdec)
                / np.hypot(pmra_error, pmdec_error)
            )

        if parallax_snr is not None:
            if parallax_snr > 5:
                flags.append("positive_parallax_gt_5sigma")
            elif parallax_snr < -5:
                flags.append("negative_parallax_outlier")

        if pm_snr is not None and pm_snr > 5:
            flags.append("proper_motion_proxy_gt_5sigma")

        if parallax_snr is None and pm_snr is None:
            flags.append("gaia_astrometry_inconclusive")

        gaia_ra = numeric(gaia["gaia_ra"])
        gaia_dec = numeric(gaia["gaia_dec"])

        match_separation = (
            separation_arcsec(ra, dec, gaia_ra, gaia_dec)
            if gaia_ra is not None and gaia_dec is not None
            else None
        )

        gaia_diagnostics.append({
            "source_id": str(gaia["gaia_source_id"]),
            "separation_arcsec": match_separation,
            "parallax_snr": parallax_snr,
            "pm_snr_proxy": pm_snr,
        })

    row = {
        # Agreed FITS-facing columns.
        # Coordinates are provisional DES positions, not Gaia replacements.
        "TARGID": f"DESDR2_{object_id}",
        "TARGNAME": f"DESDR2_{object_id}",
        "GAIA_RA": ra,
        "GAIA_DEC": dec,
        "GAIA_EPOCH": "",
        "MAG_R": numeric(des["mag_auto_r"]),
        "MAG_R_ERR": numeric(des["magerr_auto_r"]),
        "MAG_G": numeric(des["mag_auto_g"]),
        "MAG_G_ERR": numeric(des["magerr_auto_g"]),
        "MAG_I": numeric(des["mag_auto_i"]),
        "MAG_I_ERR": numeric(des["magerr_auto_i"]),
        "TARGPRIO": "",
        "TARGUSE": "T",
        "TARGCLASS": "",
        "TARGSRVY": "WS2026B2-016",
        "TARGPROG": "GW190814_HOSTS",
        "PROGTEMP": "",
        "OBSTEMP": "",

        # Provenance and review columns.
        "ORIGINAL_CATALOGUE_ID": object_id,
        "COORDINATE_SOURCE": (
            "DES DR2 ALPHAWIN_J2000/DELTAWIN_J2000; "
            "WASP astrometric convention pending"
        ),
        "MAGNITUDE_SOURCE": "DES DR2",
        "MAGNITUDE_METHOD": "MAG_AUTO; observed AB; not dereddened",
        "TARGET_CATEGORY": "",
        "PRIORITY_REASON": "",
        "PROPOSED_POINTINGS": "",
        "REVIEW_NOTES": ";".join(sorted(set(flags))),
        "REVIEW_DECISION": "pending",
        "ORIGINAL_RA": float(original["ra"]),
        "ORIGINAL_DEC": float(original["dec"]),
        "COORDINATE_DIFFERENCE_ARCSEC": coordinate_difference,
        "DES_EXTENDED_CLASS_COADD": morphology,
        "DES_SPREAD_MODEL_I": numeric(des["spread_model_i"]),
        "GAIA_MATCH_COUNT": len(gaia_matches),
        "GAIA_MATCH_REVIEW": json.dumps(gaia_diagnostics),
        "LEGACY_VIEWER_URL": (
            f"https://www.legacysurvey.org/viewer?"
            f"ra={ra}&dec={dec}&layer=ls-dr10&zoom=16"
        ),
    }

    # Keep the original quality indicators and dereddened magnitudes too.
    for band in ("g", "r", "i"):
        for column in (
            f"flags_{band}",
            f"imaflags_iso_{band}",
            f"mag_auto_{band}_dered",
        ):
            row[f"DES_{column.upper()}"] = numeric(des[column])

    review_rows.append(row)

review = pd.DataFrame(review_rows)

assert len(review) == 273
assert review["TARGID"].is_unique
assert set(review["ORIGINAL_CATALOGUE_ID"]) == set(
    targets["coadd_object_id"]
)


# ------------------------------------------------------------
# 5. Save and display the results
# ------------------------------------------------------------

output_file = OUTPUT_DIR / "GW190814_273_target_review.csv"
review.to_csv(output_file, index=False)

stellar_candidates = review[
    review["REVIEW_NOTES"].str.contains(
        "stellar_morphology_candidate"
        "|positive_parallax_gt_5sigma"
        "|proper_motion_proxy_gt_5sigma",
        regex=True,
    )
]

print(f"\nSaved {len(review)} targets to: {output_file}")
print("Targets removed: 0")
print(f"Star-candidate review list: {len(stellar_candidates)}")
print(
    "Targets with any review flag:",
    review["REVIEW_NOTES"].ne("").sum(),
)

display(
    stellar_candidates[
        [
            "ORIGINAL_CATALOGUE_ID",
            "MAG_R",
            "DES_EXTENDED_CLASS_COADD",
            "GAIA_MATCH_COUNT",
            "REVIEW_NOTES",
            "LEGACY_VIEWER_URL",
        ]
    ]
)

Loaded 273 original targets.
Querying DES DR2 and Gaia DR3...

Saved 273 targets to: GW190814_target_review/GW190814_273_target_review.csv
Targets removed: 0
Star-candidate review list: 6
Targets with any review flag: 185


,ORIGINAL_CATALOGUE_ID,MAG_R,DES_EXTENDED_CLASS_COADD,GAIA_MATCH_COUNT,REVIEW_NOTES,LEGACY_VIEWER_URL
5,1106433156,23.783106,1.0,0,stellar_morphology_candidate,https://www.legacysurvey.org/viewer?ra=13.1226...
121,1124805618,23.627028,1.0,0,stellar_morphology_candidate,https://www.legacysurvey.org/viewer?ra=13.8572...
136,1102188896,24.189056,1.0,0,stellar_morphology_candidate,https://www.legacysurvey.org/viewer?ra=11.7157...
139,1102174031,24.493460,1.0,0,stellar_morphology_candidate,https://www.legacysurvey.org/viewer?ra=12.3100...
143,1102169975,23.697649,1.0,0,stellar_morphology_candidate,https://www.legacysurvey.org/viewer?ra=12.3367...
265,1124855760,20.780970,0.0,0,extraction_flag_g;extraction_flag_i;extraction...,https://www.legacysurvey.org/viewer?ra=14.2728...


In [ ]:
from pathlib import Path
import csv
import io
import math
import time
import zipfile
from urllib.parse import urlencode
from urllib.request import urlopen
from urllib.error import HTTPError

import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, FileLink

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

# Change this if your filename is different.
INPUT_CSV = Path("/content/GW190814_target_review/GW190814_273_target_review.csv")

OUTPUT_DIR = Path("GW190814_visual_review")
CUTOUT_DIR = OUTPUT_DIR / "cutouts"
SHEET_DIR = OUTPUT_DIR / "sheets"

LAYER = "ls-dr10"
SIZE = 256
PIXEL_SCALE = 0.262       # arcsec/pixel: each thumbnail spans ~67 arcsec
COLS, ROWS = 6, 5        # 30 objects per sheet
REQUEST_PAUSE = 2        # seconds between requests
MAX_ATTEMPTS = 5

CUTOUT_DIR.mkdir(parents=True, exist_ok=True)
SHEET_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# LOAD AND CHECK THE TARGET LIST
# ------------------------------------------------------------

# Read IDs as text, preserving their exact values.
df = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)

required = {"ORIGINAL_CATALOGUE_ID", "GAIA_RA", "GAIA_DEC", "MAG_R"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["ORIGINAL_CATALOGUE_ID"] = df["ORIGINAL_CATALOGUE_ID"].str.strip()

if len(df) != 273:
    raise ValueError(f"Expected 273 targets; found {len(df)}.")

if (
    df["ORIGINAL_CATALOGUE_ID"].eq("").any()
    or df["ORIGINAL_CATALOGUE_ID"].duplicated().any()
):
    raise ValueError("Target IDs must be non-empty and unique.")

for col in ["GAIA_RA", "GAIA_DEC", "MAG_R"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

valid_coordinates = (
    df["GAIA_RA"].between(0, 360, inclusive="left")
    & df["GAIA_DEC"].between(-90, 90)
)
if not valid_coordinates.all():
    raise ValueError("Some targets have missing or invalid coordinates.")

# These columns currently contain the DES target positions,
# despite their WEAVE-required GAIA_RA / GAIA_DEC names.
print(f"Loaded {len(df)} unique targets. No cuts applied.")

# ------------------------------------------------------------
# DOWNLOAD THUMBNAILS
# Existing valid images are reused when you rerun this cell.
# ------------------------------------------------------------

def valid_image(path):
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            return im.size == (SIZE, SIZE)
    except Exception:
        return False


def fetch_cutout(row):
    obj_id = row["ORIGINAL_CATALOGUE_ID"]
    path = CUTOUT_DIR / f"{obj_id}.jpg"

    params = {
        "ra": f"{row['GAIA_RA']:.10f}",
        "dec": f"{row['GAIA_DEC']:.10f}",
        "layer": LAYER,
        "width": SIZE,
        "height": SIZE,
        "pixscale": PIXEL_SCALE,
    }
    url = (
        "https://www.legacysurvey.org/viewer/cutout.jpg?"
        + urlencode(params)
    )

    if valid_image(path):
        return path, url, "OK", ""

    last_error = ""

    for attempt in range(MAX_ATTEMPTS):
        time.sleep(REQUEST_PAUSE)

        try:
            with urlopen(url, timeout=60) as response:
                data = response.read()

            # Verify that the response is a usable image.
            with Image.open(io.BytesIO(data)) as im:
                im.load()
                if im.size != (SIZE, SIZE):
                    raise ValueError(f"Unexpected image size: {im.size}")

            # Write only after validation.
            temporary_path = path.with_suffix(".tmp")
            temporary_path.write_bytes(data)
            temporary_path.replace(path)

            return path, url, "OK", ""

        except Exception as exc:
            last_error = str(exc)

            if attempt < MAX_ATTEMPTS - 1:
                delay = min(10 * (2 ** attempt), 60)
                if isinstance(exc, HTTPError) and exc.code == 429:
                    print(
                        f"  Rate limited for {obj_id}; "
                        f"retrying in {delay} seconds.",
                        flush=True,
                    )
                time.sleep(delay)

    return path, url, "FAILED", last_error


download_records = []

for number, (_, row) in enumerate(df.iterrows(), start=1):
    path, url, status, error = fetch_cutout(row)

    download_records.append({
        "REVIEW_NUMBER": number,
        "DES_ID": row["ORIGINAL_CATALOGUE_ID"],
        "CUTOUT_FILE": str(path),
        "CUTOUT_URL": url,
        "DOWNLOAD_STATUS": status,
        "DOWNLOAD_ERROR": error,
    })

    if number % 10 == 0 or status == "FAILED" or number == len(df):
        print(
            f"{number}/{len(df)} processed — "
            f"{row['ORIGINAL_CATALOGUE_ID']}: {status}",
            flush=True,
        )

downloads = pd.DataFrame(download_records)
downloads.to_csv(OUTPUT_DIR / "download_manifest.csv", index=False)

# ------------------------------------------------------------
# FONTS
# ------------------------------------------------------------

def load_font(size):
    candidates = [
        "DejaVuSans.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "Arial.ttf",
        "arial.ttf",
        "C:/Windows/Fonts/arial.ttf",
        "/System/Library/Fonts/Supplemental/Arial.ttf",
    ]
    for candidate in candidates:
        try:
            return ImageFont.truetype(candidate, size)
        except OSError:
            pass
    return ImageFont.load_default()


title_font = load_font(26)
label_font = load_font(17)
small_font = load_font(14)

# ------------------------------------------------------------
# CREATE CONTACT SHEETS
# ------------------------------------------------------------

PER_SHEET = COLS * ROWS
N_SHEETS = math.ceil(len(df) / PER_SHEET)

MARGIN = 16
GAP = 16
HEADER_HEIGHT = 112
LABEL_HEIGHT = 60

CELL_WIDTH = SIZE + GAP
CELL_HEIGHT = SIZE + LABEL_HEIGHT

SHEET_WIDTH = 2 * MARGIN + COLS * SIZE + (COLS - 1) * GAP
SHEET_HEIGHT = HEADER_HEIGHT + ROWS * CELL_HEIGHT + MARGIN

sheet_paths = []
review_records = []

for sheet_number in range(1, N_SHEETS + 1):
    start = (sheet_number - 1) * PER_SHEET
    stop = min(start + PER_SHEET, len(df))

    sheet = Image.new("RGB", (SHEET_WIDTH, SHEET_HEIGHT), "white")
    draw = ImageDraw.Draw(sheet)

    draw.text(
        (MARGIN, 14),
        f"GW190814 — visual review — sheet {sheet_number:02d}/{N_SHEETS}",
        fill="black",
        font=title_font,
    )
    draw.text(
        (MARGIN, 51),
        f"Legacy Surveys DR10 | {SIZE} pixels | "
        f"{SIZE * PIXEL_SCALE:.1f} arcsec across",
        fill="black",
        font=label_font,
    )
    draw.text(
        (MARGIN, 78),
        "Cyan ticks mark the target position. "
        "Record decisions by review number or DES ID.",
        fill="black",
        font=small_font,
    )

    for position, index in enumerate(range(start, stop)):
        row = df.iloc[index]
        record = download_records[index]

        grid_row, grid_col = divmod(position, COLS)
        x = MARGIN + grid_col * CELL_WIDTH
        y = HEADER_HEIGHT + grid_row * CELL_HEIGHT

        if record["DOWNLOAD_STATUS"] == "OK":
            with Image.open(record["CUTOUT_FILE"]) as image:
                sheet.paste(image.convert("RGB"), (x, y))

            # Gapped crosshair: leaves the central object unobscured.
            cx = x + (SIZE - 1) / 2
            cy = y + (SIZE - 1) / 2

            for a, b in [(-14, -8), (8, 14)]:
                draw.line(
                    (cx + a, cy, cx + b, cy),
                    fill="cyan", width=1,
                )
                draw.line(
                    (cx, cy + a, cx, cy + b),
                    fill="cyan", width=1,
                )
        else:
            draw.rectangle(
                (x, y, x + SIZE - 1, y + SIZE - 1),
                fill="#eeeeee",
            )
            draw.text(
                (x + 12, y + 110),
                "DOWNLOAD FAILED",
                fill="red",
                font=label_font,
            )

        obj_id = row["ORIGINAL_CATALOGUE_ID"]
        magnitude = row["MAG_R"]
        mag_label = (
            f"{magnitude:.2f}"
            if pd.notna(magnitude) and -10 < magnitude < 90
            else "missing"
        )

        draw.text(
            (x, y + SIZE + 4),
            f"{index + 1:03d}  DES {obj_id}",
            fill="black",
            font=label_font,
        )
        draw.text(
            (x, y + SIZE + 28),
            f"r = {mag_label}   Decision: __________",
            fill="black",
            font=small_font,
        )

        viewer_url = (
            "https://www.legacysurvey.org/viewer?"
            + urlencode({
                "ra": row["GAIA_RA"],
                "dec": row["GAIA_DEC"],
                "layer": LAYER,
                "zoom": 16,
            })
        )

        review_records.append({
            "REVIEW_NUMBER": index + 1,
            "SHEET": sheet_number,
            "ROW": grid_row + 1,
            "COLUMN": grid_col + 1,
            "DES_ID": obj_id,
            "RA": row["GAIA_RA"],
            "DEC": row["GAIA_DEC"],
            "MAG_R": row["MAG_R"],
            "DECISION": "",
            "NOTES": "",
            "DOWNLOAD_STATUS": record["DOWNLOAD_STATUS"],
            "VIEWER_URL": viewer_url,
        })

    sheet_path = SHEET_DIR / f"sheet_{sheet_number:02d}.jpg"
    sheet.save(sheet_path, quality=95)
    sheet_paths.append(sheet_path)

print(f"\nCreated {len(sheet_paths)} sheets.")

# ------------------------------------------------------------
# REVIEW LOG
# Preserve decisions if you have already edited this log.
# ------------------------------------------------------------

review = pd.DataFrame(review_records)
log_path = OUTPUT_DIR / "review_log.csv"

if log_path.exists():
    previous = pd.read_csv(
        log_path, dtype=str, keep_default_na=False
    )
    if {"DES_ID", "DECISION", "NOTES"}.issubset(previous.columns):
        if previous["DES_ID"].duplicated().any():
            raise ValueError("Existing review log contains duplicate IDs.")
        previous = previous.set_index("DES_ID")
        for column in ["DECISION", "NOTES"]:
            review[column] = (
                review["DES_ID"].map(previous[column]).fillna("")
            )

review.to_csv(log_path, index=False)

# ------------------------------------------------------------
# COMBINED PDF AND ZIP
# ------------------------------------------------------------

pdf_path = OUTPUT_DIR / "GW190814_273_review_sheets.pdf"
pdf_images = []

try:
    for path in sheet_paths:
        with Image.open(path) as image:
            pdf_images.append(image.convert("RGB"))

    pdf_images[0].save(
        pdf_path,
        "PDF",
        save_all=True,
        append_images=pdf_images[1:],
        resolution=150.0,
    )
finally:
    for image in pdf_images:
        image.close()

failed = downloads[downloads["DOWNLOAD_STATUS"] != "OK"]

readme = (
    "GW190814: visual review of all 273 targets.\n"
    "Original CSV order retained; no exclusions or magnitude cuts.\n"
    "Ten sheets: nine with 30 objects, the last with 3.\n"
    "Images: Legacy Surveys DR10, 256 x 256 pixels.\n"
    "Scale fixed at 0.262 arcsec/pixel (~67 arcsec across).\n"
    "Cyan ticks indicate the supplied DES target position.\n"
    "r magnitudes are the DES magnitudes from the input review CSV.\n"
    "Individual JPEGs in cutouts/ have no added centre markers.\n"
    "Suggested decisions: KEEP, POSSIBLE_STAR, BLEND, ARTEFACT, UNCERTAIN.\n"
    "Visual classifications remain provisional.\n"
    f"Failed downloads: {len(failed)}. Rerun the cell to retry them.\n"
)
(OUTPUT_DIR / "README.txt").write_text(readme, encoding="utf-8")

zip_path = Path("GW190814_visual_review.zip")

# Include exactly this run's target images and review products.
files_to_zip = (
    [Path(r["CUTOUT_FILE"]) for r in download_records
     if r["DOWNLOAD_STATUS"] == "OK"]
    + sheet_paths
    + [
        log_path,
        pdf_path,
        OUTPUT_DIR / "download_manifest.csv",
        OUTPUT_DIR / "README.txt",
    ]
)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in files_to_zip:
        archive.write(path, path.relative_to(OUTPUT_DIR))
    archive.write(INPUT_CSV, "input_target_review.csv")

print(f"\nSuccessful thumbnails: {len(df) - len(failed)}/{len(df)}")
print(f"Review log: {log_path}")
print("No objects have been removed.")

if len(failed):
    print("\nFailed downloads — these have labelled placeholders:")
    display(failed[["REVIEW_NUMBER", "DES_ID", "DOWNLOAD_ERROR"]])

# Show the first sheet.
display(Image.open(sheet_paths[0]))

# Download links for Jupyter.
display(FileLink(str(pdf_path)))
display(FileLink(str(log_path)))
display(FileLink(str(zip_path)))

In [9]:
%pip -q install pyvo astropy

from pathlib import Path
import io
import json
import time
import zipfile

import numpy as np
import pandas as pd
import pyvo
from astropy.coordinates import SkyCoord
import astropy.units as u
from IPython.display import display, FileLink


# ============================================================
# INPUT / OUTPUT
# Reuses INPUT_CSV from your earlier notebook cell.
# ============================================================

if "INPUT_CSV" not in globals():
    raise ValueError(
        "Run the earlier cell that sets INPUT_CSV to your "
        "GW190814_273_target_review.csv file, then run this cell."
    )

INPUT_CSV = Path(INPUT_CSV)

if not INPUT_CSV.is_file():
    raise FileNotFoundError(f"Cannot find the input CSV: {INPUT_CSV}")

OUT = Path("GW190814_15_object_catalogue_checks_v2")
OUT.mkdir(exist_ok=True)

RADIUS_ARCSEC = 3.0

TARGETS = [
    (6,   "1106433156", "faint_morphology_candidate"),
    (12,  "1113379333", "visual_possible_star"),
    (30,  "1123263115", "visual_possible_star"),
    (32,  "1113366245", "visual_uncertain"),
    (83,  "1095490490", "visual_uncertain"),
    (98,  "1113986212", "visual_uncertain"),
    (121, "1112778820", "visual_uncertain"),
    (122, "1124805618", "faint_morphology_candidate"),
    (123, "1095342910", "visual_possible_star"),
    (137, "1102188896", "faint_morphology_candidate"),
    (140, "1102174031", "faint_morphology_candidate"),
    (144, "1102169975", "faint_morphology_candidate"),
    (173, "1098495648", "visual_probable_star"),
    (185, "1107630407", "visual_probable_star"),
    (266, "1124855760", "faint_morphology_candidate"),
]

seed = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)

required = {
    "ORIGINAL_CATALOGUE_ID", "GAIA_RA", "GAIA_DEC", "MAG_R"
}
missing = required - set(seed.columns)
if missing:
    raise ValueError(f"Input CSV is missing columns: {sorted(missing)}")

seed["ORIGINAL_CATALOGUE_ID"] = (
    seed["ORIGINAL_CATALOGUE_ID"].str.strip()
)

if seed["ORIGINAL_CATALOGUE_ID"].duplicated().any():
    raise ValueError("Input CSV contains duplicate DES IDs.")

wanted = pd.DataFrame(
    TARGETS, columns=["IMAGE_NUMBER", "DES_ID", "REVIEW_REASON"]
)

targets = wanted.merge(
    seed,
    left_on="DES_ID",
    right_on="ORIGINAL_CATALOGUE_ID",
    how="left",
    validate="one_to_one",
    indicator=True,
)

if not targets["_merge"].eq("both").all():
    missing_ids = targets.loc[
        targets["_merge"] != "both", "DES_ID"
    ].tolist()
    raise ValueError(f"Missing target IDs: {missing_ids}")

targets = targets.drop(columns="_merge")
targets["RA_QUERY"] = pd.to_numeric(targets["GAIA_RA"], errors="raise")
targets["DEC_QUERY"] = pd.to_numeric(targets["GAIA_DEC"], errors="raise")

if not (
    targets["RA_QUERY"].between(0, 360, inclusive="left")
    & targets["DEC_QUERY"].between(-90, 90)
).all():
    raise ValueError("Invalid target coordinates.")

written_files = set()


def save_csv(frame, filename):
    frame.to_csv(OUT / filename, index=False)
    written_files.add(filename)


save_csv(targets, "selected_15_targets.csv")

print(f"Input: {INPUT_CSV}", flush=True)
print(f"Selected {len(targets)} targets for catalogue checks.", flush=True)


# ============================================================
# QUERY HELPERS
# ============================================================

service = pyvo.dal.TAPService("https://datalab.noirlab.edu/tap")

query_log = []
schema_log = {}
schema_cache = {}
status_records = []


def save_logs():
    (OUT / "query_log.json").write_text(
        json.dumps(query_log, indent=2), encoding="utf-8"
    )
    (OUT / "column_availability.json").write_text(
        json.dumps(schema_log, indent=2), encoding="utf-8"
    )
    written_files.update({
        "query_log.json", "column_availability.json"
    })


def query(sql, label, attempts=3):
    """Preserve catalogue IDs as strings; log successful and failed queries."""
    error = ""

    for attempt in range(attempts):
        try:
            result = service.search(sql, maxrec=10000)
            table = result.to_table()

            # Avoid converting large integer identifiers into floats.
            buffer = io.StringIO()
            table.write(buffer, format="ascii.csv")
            buffer.seek(0)

            frame = pd.read_csv(
                buffer, dtype=str, keep_default_na=False
            )
            frame.columns = frame.columns.str.lower()

            query_log.append({
                "label": label,
                "status": "OK",
                "rows": len(frame),
                "sql": sql,
            })
            save_logs()
            return frame

        except Exception as exc:
            error = str(exc)

            # Retrying a syntax error will not repair the query.
            permanent_error = any(
                phrase in error.lower()
                for phrase in [
                    "does not exist",
                    "syntax error",
                    "parseexception",
                    "not found in tapschema",
                ]
            )

            if permanent_error or attempt == attempts - 1:
                break

            print(
                f"{label}: request failed; retrying "
                f"({attempt + 1}/{attempts})...",
                flush=True,
            )
            time.sleep(5 * (attempt + 1))

    query_log.append({
        "label": label,
        "status": "FAILED",
        "error": error,
        "sql": sql,
    })
    save_logs()
    raise RuntimeError(f"{label}: {error}")


def available_columns(table_name):
    if table_name not in schema_cache:
        metadata = query(
            "SELECT column_name FROM TAP_SCHEMA.columns "
            f"WHERE table_name = '{table_name}'",
            f"schema:{table_name}",
        )
        columns = {
            name.lower()
            for name in metadata.get(
                "column_name", pd.Series(dtype=str)
            )
        }
        if not columns:
            raise RuntimeError(f"No column metadata for {table_name}")
        schema_cache[table_name] = columns

    return schema_cache[table_name]


def choose_columns(table_name, desired, required):
    available = available_columns(table_name)
    missing_required = set(required) - available

    if missing_required:
        raise RuntimeError(
            f"{table_name}: missing required columns "
            f"{sorted(missing_required)}"
        )

    selected = [name for name in desired if name in available]
    schema_log[table_name] = {
        "selected": selected,
        "unavailable_optional": [
            name for name in desired if name not in available
        ],
    }
    save_logs()
    return selected


def select_sql(columns):
    return ", ".join(f'"{name}"' for name in columns)


def cone_sql(table_name, columns, ra, dec, top=None):
    # NOIRLab-supported cone-search syntax.
    limit = f"TOP {int(top)} " if top is not None else ""
    return f"""
    SELECT {limit}{select_sql(columns)}
    FROM {table_name}
    WHERE 't' = q3c_radial_query(
        ra, dec,
        {float(ra):.10f}, {float(dec):.10f},
        {RADIUS_ARCSEC / 3600:.12f}
    )
    """


# ============================================================
# CATALOGUES AND REQUESTED MEASUREMENTS
# ============================================================

des_columns = [
    "coadd_object_id", "ra", "dec", "extended_class_coadd",
]

for band in ["g", "r", "i", "z"]:
    des_columns += [
        f"mag_auto_{band}", f"magerr_auto_{band}",
        f"mag_psf_{band}", f"magerr_psf_{band}",
        f"wavg_mag_psf_{band}", f"wavg_magerr_psf_{band}",
        f"spread_model_{band}", f"spreaderr_model_{band}",
        f"wavg_spread_model_{band}", f"wavg_spreaderr_model_{band}",
        f"class_star_{band}", f"flags_{band}",
        f"imaflags_iso_{band}",
    ]

bulk_specs = [
    ("des", "des_dr2.main", des_columns),
    ("des_y6", "des_dr2.y6_gold", [
        "coadd_object_id", "dnf_z",
        "wavg_spread_model_z", "wavg_spreaderr_model_z",
        "spread_model_z", "spreaderr_model_z",
    ]),
]

catalogues = {
    "legacy": {
        "table": "ls_dr10.tractor",
        "columns": [
            "ls_id", "release", "brickid", "brickname", "objid",
            "ra", "dec", "brick_primary", "type",
            "shape_r", "shape_r_ivar", "shape_e1", "shape_e2",
            "sersic", "sersic_ivar",
            "ref_cat", "ref_id", "maskbits", "fitbits",
            "flux_g", "flux_r", "flux_i", "flux_z",
            "flux_ivar_g", "flux_ivar_r", "flux_ivar_i", "flux_ivar_z",
            "fracflux_g", "fracflux_r", "fracflux_z",
            "fracmasked_g", "fracmasked_r", "fracmasked_z",
            "fracin_g", "fracin_r", "fracin_z",
            "rchisq_g", "rchisq_r", "rchisq_z",
            "nobs_g", "nobs_r", "nobs_z",
        ],
    },
    "gaia": {
        "table": "gaia_dr3.gaia_source",
        "columns": [
            "source_id", "ra", "dec", "ref_epoch",
            "ra_error", "dec_error",
            "parallax", "parallax_error", "parallax_over_error",
            "pmra", "pmra_error", "pmdec", "pmdec_error",
            "pmra_pmdec_corr", "astrometric_params_solved",
            "ruwe", "astrometric_excess_noise",
            "astrometric_excess_noise_sig", "visibility_periods_used",
            "duplicated_source", "ipd_frac_multi_peak",
            "phot_g_mean_mag", "phot_bp_mean_mag", "phot_rp_mean_mag",
            "bp_rp", "phot_bp_rp_excess_factor",
            "classprob_dsc_combmod_star",
            "classprob_dsc_combmod_galaxy",
            "classprob_dsc_combmod_quasar",
            "in_galaxy_candidates", "in_qso_candidates",
            "non_single_star",
        ],
    },
    "sdss_spectra": {
        "table": "sdss_dr17.specobj",
        "columns": [
            "specobjid", "ra", "dec",
            "class", "subclass", "z", "zerr", "zwarning",
            "snmedian", "survey", "programname",
            "plate", "mjd", "fiberid",
        ],
    },
}


# ============================================================
# PREFLIGHT: STOP EARLY IF THE SEARCH SYNTAX FAILS
# Zero matches is a successful query, not a failure.
# ============================================================

print("\nChecking catalogue access and search syntax...", flush=True)

test_target = targets.iloc[0]
prepared_columns = {}

for key, config in catalogues.items():
    try:
        columns = choose_columns(
            config["table"], config["columns"], ["ra", "dec"]
        )
        prepared_columns[key] = columns

        query(
            cone_sql(
                config["table"],
                columns,
                test_target["RA_QUERY"],
                test_target["DEC_QUERY"],
                top=1,
            ),
            f"preflight:{key}",
            attempts=1,
        )

    except Exception as exc:
        raise RuntimeError(
            f"Preflight failed for {key}. "
            "Stopped before the full set of searches. "
            f"Please paste this error into the chat:\n{exc}"
        ) from exc

    print(f"Preflight OK: {key}", flush=True)


# ============================================================
# DES: EXACT OBJECT-ID QUERIES
# ============================================================

print("\nFetching DES measurements...", flush=True)

ids_sql = ", ".join(targets["DES_ID"])
bulk_results = {}

for key, table_name, desired in bulk_specs:
    try:
        columns = choose_columns(
            table_name, desired, ["coadd_object_id"]
        )

        result = query(
            f"SELECT {select_sql(columns)} FROM {table_name} "
            f"WHERE coadd_object_id IN ({ids_sql})",
            key,
        )

        if result["coadd_object_id"].duplicated().any():
            raise ValueError(f"{key}: unexpected duplicate DES IDs")

        save_csv(result, f"{key}_measurements.csv")
        bulk_results[key] = result

        found_ids = set(result["coadd_object_id"])

        for des_id in targets["DES_ID"]:
            status_records.append({
                "DES_ID": des_id,
                "CATALOGUE": key,
                "STATUS": "OK",
                "MATCH_COUNT": int(des_id in found_ids),
                "ERROR": "",
            })

        print(f"{key}: returned {len(result)} objects", flush=True)

    except Exception as exc:
        for des_id in targets["DES_ID"]:
            status_records.append({
                "DES_ID": des_id,
                "CATALOGUE": key,
                "STATUS": "FAILED",
                "MATCH_COUNT": "",
                "ERROR": str(exc),
            })

        print(f"{key}: FAILED — {exc}", flush=True)


# ============================================================
# POSITIONAL SEARCHES
# Retain ALL matches within 3 arcseconds.
# ============================================================

all_matches = {}

for key, config in catalogues.items():
    columns = prepared_columns[key]
    pieces = []

    print(f"\nSearching {key}...", flush=True)

    for number, (_, target) in enumerate(targets.iterrows(), 1):
        ra = float(target["RA_QUERY"])
        dec = float(target["DEC_QUERY"])

        sql = cone_sql(config["table"], columns, ra, dec)

        try:
            result = query(sql, f"{key}:{target['DES_ID']}")

            if not result.empty:
                source_positions = SkyCoord(
                    pd.to_numeric(result["ra"]).to_numpy() * u.deg,
                    pd.to_numeric(result["dec"]).to_numpy() * u.deg,
                )
                target_position = SkyCoord(ra * u.deg, dec * u.deg)

                result["separation_arcsec"] = (
                    target_position.separation(source_positions).arcsec
                )
                result = result.sort_values("separation_arcsec")
                result.insert(0, "des_id", target["DES_ID"])
                result.insert(1, "image_number", target["IMAGE_NUMBER"])
                pieces.append(result)

            status_records.append({
                "DES_ID": target["DES_ID"],
                "CATALOGUE": key,
                "STATUS": "OK",
                "MATCH_COUNT": len(result),
                "ERROR": "",
            })

            print(
                f"{key}: {number}/{len(targets)} — image "
                f"{int(target['IMAGE_NUMBER']):03d}: "
                f"{len(result)} matches",
                flush=True,
            )

        except Exception as exc:
            status_records.append({
                "DES_ID": target["DES_ID"],
                "CATALOGUE": key,
                "STATUS": "FAILED",
                "MATCH_COUNT": "",
                "ERROR": str(exc),
            })

            print(f"{key}: FAILED — {exc}", flush=True)

        # Save progress after every target.
        save_csv(pd.DataFrame(status_records), "query_status.csv")

        if pieces:
            save_csv(
                pd.concat(pieces, ignore_index=True),
                f"{key}_all_matches.csv",
            )

        time.sleep(0.5)

    combined = (
        pd.concat(pieces, ignore_index=True)
        if pieces
        else pd.DataFrame(columns=[
            "des_id", "image_number", *columns, "separation_arcsec"
        ])
    )

    all_matches[key] = combined
    save_csv(combined, f"{key}_all_matches.csv")


# ============================================================
# SUMMARY
# Nearest match is shown for convenience, not confirmed identity.
# ============================================================

summary = targets[
    ["IMAGE_NUMBER", "DES_ID", "REVIEW_REASON",
     "RA_QUERY", "DEC_QUERY", "MAG_R"]
].copy()

for key, result in bulk_results.items():
    renamed = result.rename(columns={
        col: "DES_ID" if col == "coadd_object_id" else f"{key}_{col}"
        for col in result.columns
    })

    summary = summary.merge(
        renamed, on="DES_ID", how="left", validate="one_to_one"
    )

statuses = pd.DataFrame(status_records)
save_csv(statuses, "query_status.csv")

for key in ["des", "des_y6", *catalogues.keys()]:
    status = statuses[statuses["CATALOGUE"] == key][
        ["DES_ID", "STATUS", "MATCH_COUNT"]
    ].rename(columns={
        "STATUS": f"{key}_query_status",
        "MATCH_COUNT": f"{key}_match_count",
    })

    summary = summary.merge(
        status, on="DES_ID", how="left", validate="one_to_one"
    )

for key, matches in all_matches.items():
    if matches.empty:
        continue

    nearest = (
        matches.sort_values("separation_arcsec")
        .drop_duplicates("des_id")
        .drop(columns="image_number")
    )

    nearest = nearest.rename(columns={
        col: "DES_ID" if col == "des_id" else f"{key}_nearest_{col}"
        for col in nearest.columns
    })

    summary = summary.merge(
        nearest, on="DES_ID", how="left", validate="one_to_one"
    )

# Per-component astrometric significance: evidence for review only.
for value, error in [
    ("parallax", "parallax_error"),
    ("pmra", "pmra_error"),
    ("pmdec", "pmdec_error"),
]:
    value_col = f"gaia_nearest_{value}"
    error_col = f"gaia_nearest_{error}"

    if value_col in summary and error_col in summary:
        measurement = pd.to_numeric(summary[value_col], errors="coerce")
        uncertainty = pd.to_numeric(summary[error_col], errors="coerce")

        valid = (
            np.isfinite(measurement)
            & np.isfinite(uncertainty)
            & (uncertainty > 0)
        )

        summary[f"gaia_nearest_{value}_snr"] = (
            measurement.where(valid) / uncertainty.where(valid)
        )

summary["FINAL_DECISION"] = ""
summary["FOLLOWUP_NOTES"] = ""

save_csv(summary, "catalogue_evidence_summary.csv")
save_logs()

(OUT / "README.txt").write_text(
    "Evidence review for 15 GW190814 targets. No objects excluded.\n"
    "DES queries use exact object IDs.\n"
    "Legacy, Gaia and SDSS searches retain all matches within 3 arcsec.\n"
    "Cone searches use NOIRLab's q3c_radial_query function.\n"
    "The summary shows the nearest match for convenience only.\n"
    "Check separations and competing matches before assigning identity.\n"
    "Legacy brick_primary and ref_cat/ref_id are retained where available.\n"
    "Gaia positions are searched as catalogued; no epoch propagation.\n"
    "Missing Gaia astrometry does not establish a galaxy classification.\n"
    "Morphology/model labels are evidence, not definitive identities.\n"
    "Raw catalogue sentinel values such as -99 are preserved, not cleaned.\n"
    "SDSS is not an exhaustive search of existing spectroscopy.\n"
    "FAILED queries must not be interpreted as zero matches.\n"
    "See column_availability.json for fields absent from hosted tables.\n"
    "These outputs are for review, not a finished WASP upload catalogue.\n",
    encoding="utf-8",
)
written_files.add("README.txt")


# ============================================================
# ZIP AND DISPLAY
# ============================================================

zip_path = Path("GW190814_15_object_catalogue_checks_v2.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for filename in sorted(written_files):
        archive.write(OUT / filename, filename)

display_columns = [
    "IMAGE_NUMBER", "DES_ID", "REVIEW_REASON", "MAG_R",
    "legacy_query_status", "legacy_match_count",
    "legacy_nearest_type", "legacy_nearest_shape_r",
    "legacy_nearest_separation_arcsec",
    "gaia_query_status", "gaia_match_count",
    "gaia_nearest_parallax_snr",
    "gaia_nearest_pmra_snr", "gaia_nearest_pmdec_snr",
    "sdss_spectra_query_status",
    "sdss_spectra_nearest_class", "sdss_spectra_nearest_z",
    "sdss_spectra_nearest_zwarning",
]

display(summary[[c for c in display_columns if c in summary]])

failed = statuses[statuses["STATUS"] == "FAILED"]

print(f"\nFinished. Failed target/catalogue checks: {len(failed)}")
print(f"Original input CSV unchanged: {len(seed)} targets.")
print(f"ZIP created: {zip_path.resolve()}")

if not failed.empty:
    print("\nSome checks failed; their missing results are inconclusive.")
    display(failed[["DES_ID", "CATALOGUE", "ERROR"]])

print("\nUpload the v2 ZIP here so we can review the evidence.")

display(FileLink(str(zip_path)))

# Automatically offer the ZIP for download when running in Colab.
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(zip_path))

Input: /content/GW190814_target_review/GW190814_273_target_review.csv
Selected 15 targets for catalogue checks.

Checking catalogue access and search syntax...
Preflight OK: legacy
Preflight OK: gaia
Preflight OK: sdss_spectra

Fetching DES measurements...
des: returned 15 objects
des_y6: returned 15 objects

Searching legacy...
legacy: 1/15 — image 006: 1 matches
legacy: 2/15 — image 012: 1 matches
legacy: 3/15 — image 030: 1 matches
legacy: 4/15 — image 032: 2 matches
legacy: 5/15 — image 083: 3 matches
legacy: 6/15 — image 098: 1 matches
legacy: 7/15 — image 121: 3 matches
legacy: 8/15 — image 122: 2 matches
legacy: 9/15 — image 123: 3 matches
legacy: 10/15 — image 137: 1 matches
legacy: 11/15 — image 140: 1 matches
legacy: 12/15 — image 144: 1 matches
legacy: 13/15 — image 173: 1 matches
legacy: 14/15 — image 185: 1 matches
legacy: 15/15 — image 266: 1 matches

Searching gaia...
gaia: 1/15 — image 006: 0 matches
gaia: 2/15 — image 012: 1 matches
gaia: 3/15 — image 030: 1 matches
ga

,IMAGE_NUMBER,DES_ID,REVIEW_REASON,MAG_R,legacy_query_status,legacy_match_count,legacy_nearest_type,legacy_nearest_shape_r,legacy_nearest_separation_arcsec,gaia_query_status,gaia_match_count,gaia_nearest_parallax_snr,gaia_nearest_pmra_snr,gaia_nearest_pmdec_snr,sdss_spectra_query_status
0,6,1106433156,faint_morphology_candidate,23.783106,OK,1,PSF,0.0,0.746459,OK,0,NaN,NaN,NaN,OK
1,12,1113379333,visual_possible_star,17.370201,OK,1,PSF,0.0,0.235831,OK,1,NaN,NaN,NaN,OK
2,30,1123263115,visual_possible_star,15.536342,OK,1,PSF,0.0,0.134568,OK,1,NaN,NaN,NaN,OK
3,32,1113366245,visual_uncertain,17.225994,OK,2,REX,1.416272521018982,0.395830,OK,1,NaN,NaN,NaN,OK
4,83,1095490490,visual_uncertain,17.430468,OK,3,SER,0.3231479823589325,0.004788,OK,1,-0.281531,-0.016233,0.100185,OK
5,98,1113986212,visual_uncertain,19.581577,OK,1,SER,0.35886457562446594,0.013933,OK,1,2.158935,1.583638,0.296550,OK
6,121,1112778820,visual_uncertain,18.096556,OK,3,REX,0.9091323614120483,0.037476,OK,1,NaN,NaN,NaN,OK
7,122,1124805618,faint_morphology_candidate,23.627028,OK,2,PSF,0.0,0.126328,OK,0,NaN,NaN,NaN,OK
8,123,1095342910,visual_possible_star,17.746683,OK,3,REX,0.34388619661331177,0.009555,OK,1,NaN,NaN,NaN,OK
9,137,1102188896,faint_morphology_candidate,24.189056,OK,1,PSF,0.0,0.033180,OK,0,NaN,NaN,NaN,OK



Finished. Failed target/catalogue checks: 0
Original input CSV unchanged: 273 targets.
ZIP created: /content/GW190814_15_object_catalogue_checks_v2.zip

Upload the v2 ZIP here so we can review the evidence.


/content/GW190814_15_object_catalogue_checks_v2.zip

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
from pathlib import Path
import csv
import hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import display, FileLink


# ============================================================
# GW190814: COMPACT 269-TARGET LIST FOR MARK
#
# Starting sample: original 273-target review CSV.
# Decision: exclude four probable stellar contaminants.
# No magnitude cut, coordinate changes, or new field selection.
#
# Reuses INPUT_CSV from your earlier notebook cells.
# ============================================================

if "INPUT_CSV" not in globals():
    raise ValueError(
        "First run the notebook cell that sets INPUT_CSV "
        "to your original 273-target review CSV."
    )

source_path = Path(INPUT_CSV)

if not source_path.is_file():
    raise FileNotFoundError(f"Cannot find: {source_path}")

output_dir = Path("GW190814_Mark_review")
output_dir.mkdir(exist_ok=True)

main_path = output_dir / "GW190814_269_targets_for_Mark.csv"
excluded_path = output_dir / "GW190814_four_excluded_targets.csv"
methods_path = output_dir / "GW190814_selection_notes.txt"


# ============================================================
# LOAD AND VALIDATE THE ORIGINAL SAMPLE
# Read identifiers and measurements as text to preserve values.
# ============================================================

df = pd.read_csv(
    source_path,
    dtype=str,
    keep_default_na=False,
)

required_columns = {
    "ORIGINAL_CATALOGUE_ID",
    "GAIA_RA",
    "GAIA_DEC",
    "MAG_R",
    "MAG_R_ERR",
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Input CSV is missing columns: {sorted(missing_columns)}"
    )

if len(df) != 273:
    raise ValueError(
        f"Expected the original 273-target CSV, but found {len(df)} rows. "
        "Do not use an already-filtered CSV as INPUT_CSV."
    )

df["ORIGINAL_CATALOGUE_ID"] = (
    df["ORIGINAL_CATALOGUE_ID"].str.strip()
)

if df["ORIGINAL_CATALOGUE_ID"].eq("").any():
    raise ValueError("Input contains blank DES identifiers.")

if df["ORIGINAL_CATALOGUE_ID"].duplicated().any():
    raise ValueError("Input contains duplicate DES identifiers.")

ra = pd.to_numeric(df["GAIA_RA"], errors="raise")
dec = pd.to_numeric(df["GAIA_DEC"], errors="raise")

if not (
    np.isfinite(ra)
    & np.isfinite(dec)
    & ra.between(0, 360, inclusive="left")
    & dec.between(-90, 90)
).all():
    raise ValueError("Invalid coordinates found.")

# These are original thumbnail numbers, not new 1–269 numbers.
df.insert(0, "IMAGE_NUMBER", range(1, len(df) + 1))

# Known associations from the original review sheets.
# This guards against accidentally using a reordered input file.
known_image_ids = {
    6:   "1106433156",
    12:  "1113379333",
    30:  "1123263115",
    32:  "1113366245",
    83:  "1095490490",
    98:  "1113986212",
    121: "1112778820",
    122: "1124805618",
    123: "1095342910",
    137: "1102188896",
    140: "1102174031",
    144: "1102169975",
    173: "1098495648",
    185: "1107630407",
    266: "1124855760",
}

for image_number, expected_id in known_image_ids.items():
    actual_id = df.iloc[image_number - 1]["ORIGINAL_CATALOGUE_ID"]
    if actual_id != expected_id:
        raise ValueError(
            f"Thumbnail numbering mismatch at image {image_number}: "
            f"expected DES {expected_id}, found {actual_id}. "
            "Use the original CSV in the order used for the review sheets."
        )


# ============================================================
# EXCLUSION DECISION
# Exclude by stable DES identifier, not by row position.
# These are probable stars, not spectroscopically confirmed stars.
# ============================================================

exclusion_reasons = {
    "1113379333": (
        "Probable stellar contaminant: visually suspicious; "
        "Gaia DSC star score 0.9965; Legacy Gaia-point-source treatment. "
        "No usable Gaia parallax/proper motion in retrieved results."
    ),
    "1123263115": (
        "Probable stellar contaminant: visually suspicious; "
        "Gaia DSC star score 0.9981; Legacy Gaia-point-source treatment. "
        "No usable Gaia parallax/proper motion in retrieved results."
    ),
    "1098495648": (
        "Probable stellar contaminant: visually judged strongly star-like; "
        "Gaia DSC star score 0.7394; Legacy Gaia-point-source treatment. "
        "No usable Gaia parallax/proper motion in retrieved results."
    ),
    "1107630407": (
        "Probable stellar contaminant: visually judged probably stellar; "
        "Gaia DSC star score 0.8514; Legacy Gaia-point-source treatment. "
        "No usable Gaia parallax/proper motion in retrieved results."
    ),
}

excluded_ids = set(exclusion_reasons)

if not excluded_ids.issubset(set(df["ORIGINAL_CATALOGUE_ID"])):
    raise ValueError("One or more exclusion IDs are absent from the input.")

exclude_mask = df["ORIGINAL_CATALOGUE_ID"].isin(excluded_ids)


# ============================================================
# COMPACT REVIEW TABLE
#
# Source columns GAIA_RA/GAIA_DEC currently contain DES-derived
# coordinates in this working catalogue; rename them neutrally.
# Preserve the original coordinate and magnitude values.
# ============================================================

compact = df[
    [
        "IMAGE_NUMBER",
        "ORIGINAL_CATALOGUE_ID",
        "GAIA_RA",
        "GAIA_DEC",
        "MAG_R",
        "MAG_R_ERR",
    ]
].rename(columns={
    "ORIGINAL_CATALOGUE_ID": "DES_ID",
    "GAIA_RA": "RA_DEG",
    "GAIA_DEC": "DEC_DEG",
    "MAG_R": "DES_R_MAG",
    "MAG_R_ERR": "DES_R_MAG_ERR",
})

compact["REVIEW_STATUS"] = "RETAIN"
compact["REVIEW_NOTE"] = ""


# Selected interpretation notes from visual review and the
# successful 15-object catalogue check.
review_notes = {
    6: (
        "RETAIN_WITH_CAVEAT",
        "Faint; Legacy point-source model; no Gaia match within 3 arcsec. "
        "Legacy separation 0.75 arcsec; association/photometry uncertain."
    ),
    32: (
        "RETAIN",
        "Galaxy supported by Legacy extended model and Gaia galaxy score 0.9918."
    ),
    83: (
        "RETAIN",
        "Galaxy supported by Legacy extended model and Gaia galaxy score 0.9948."
    ),
    98: (
        "FOLLOW_UP",
        "Gaia quasar score 0.9989; Legacy extended model. "
        "Check AGN identity/redshift and suitability as a candidate host."
    ),
    121: (
        "RETAIN",
        "Galaxy supported by Legacy extended model and Gaia galaxy score ~1."
    ),
    122: (
        "RETAIN_WITH_CAVEAT",
        "Faint; nearest Legacy source is point-like; "
        "extended neighbour 2.38 arcsec away. Possible blending."
    ),
    123: (
        "FOLLOW_UP",
        "Conflicting classification: Gaia star score 0.9792 but also "
        "a Gaia galaxy candidate; Legacy and DES favour extended morphology."
    ),
    137: (
        "RETAIN_WITH_CAVEAT",
        "Faint; Legacy point-source model; no Gaia match within 3 arcsec. "
        "Stellar identity unconfirmed."
    ),
    140: (
        "RETAIN_WITH_CAVEAT",
        "Faint; Legacy model forced to point-source type. "
        "Stellar identity unconfirmed."
    ),
    144: (
        "RETAIN_WITH_CAVEAT",
        "Faint; Legacy model forced to point-source type. "
        "Stellar identity unconfirmed."
    ),
    266: (
        "FOLLOW_UP",
        "Legacy extended model supports retention. "
        "Photometry discrepancy: DES r=20.78 versus Legacy r~23.39; "
        "resolve before assigning observing priority."
    ),
}

# Visual image-quality concerns recorded by Kelly.
# These do not trigger exclusion.
image_quality_notes = {
    43:  "Legacy image: green band away from target centre; retained.",
    73:  "Legacy image: washed-out appearance; retained pending photometry review.",
    74:  "Legacy image: green band just below target; retained.",
    142: "Legacy image: green band away from target; retained.",
    143: "Legacy image: orange stripes away from target; retained.",
    189: "Legacy image: streak away from target; retained.",
    212: "Legacy image: nearby-star streak; retained.",
    263: "Legacy image: streak; retained.",
    264: "Legacy image: streak; retained.",
}

for image_number, (status, note) in review_notes.items():
    match = compact["IMAGE_NUMBER"].eq(image_number)
    compact.loc[match, "REVIEW_STATUS"] = status
    compact.loc[match, "REVIEW_NOTE"] = note

for image_number, note in image_quality_notes.items():
    match = compact["IMAGE_NUMBER"].eq(image_number)

    for index in compact.index[match]:
        previous = compact.at[index, "REVIEW_NOTE"]
        compact.at[index, "REVIEW_NOTE"] = (
            f"{previous} {note}".strip()
        )
        if compact.at[index, "REVIEW_STATUS"] == "RETAIN":
            compact.at[index, "REVIEW_STATUS"] = "RETAIN_WITH_CAVEAT"


# ============================================================
# APPLY THE FOUR EXCLUSIONS AND CHECK THE RESULT
# ============================================================

retained = compact.loc[~exclude_mask].copy()
excluded = compact.loc[exclude_mask].copy()

excluded["REVIEW_STATUS"] = "EXCLUDE_PROBABLE_STAR"
excluded["REVIEW_NOTE"] = excluded["DES_ID"].map(exclusion_reasons)

assert len(retained) == 269, "Unexpected retained count."
assert len(excluded) == 4, "Unexpected excluded count."
assert retained["DES_ID"].is_unique
assert set(excluded["DES_ID"]) == excluded_ids
assert not set(retained["DES_ID"]) & excluded_ids
assert set(retained["DES_ID"]) | excluded_ids == set(compact["DES_ID"])

# Preserve original order and thumbnail numbers, including gaps.
# No hard magnitude threshold is applied.
retained.to_csv(main_path, index=False, quoting=csv.QUOTE_MINIMAL)
excluded.to_csv(excluded_path, index=False, quoting=csv.QUOTE_MINIMAL)

# Verify the saved CSV has the expected rows and exact identifiers.
saved = pd.read_csv(main_path, dtype=str, keep_default_na=False)
assert len(saved) == 269
assert saved["DES_ID"].tolist() == retained["DES_ID"].tolist()

for column in ["RA_DEG", "DEC_DEG", "DES_R_MAG", "DES_R_MAG_ERR"]:
    assert saved[column].tolist() == retained[column].tolist()


# ============================================================
# TRACEABILITY / METHODS NOTE
# ============================================================

source_hash = hashlib.sha256(source_path.read_bytes()).hexdigest()
created_utc = datetime.now(timezone.utc).isoformat()

methods = [
    "GW190814: compact candidate-target list for supervisor sense check",
    f"Created UTC: {created_utc}",
    f"Input file: {source_path.resolve()}",
    f"Input SHA256: {source_hash}",
    "",
    "Selection:",
    "Started with 273 targets in the original thumbnail-sheet order.",
    "Removed four probable stellar contaminants by exact DES catalogue ID.",
    "Retained 269 targets, including unresolved classifications.",
    "No magnitude cut, coordinate changes, spatial cut, or field-centre "
    "optimisation was applied.",
    "The resulting count is not a criterion for stellar classification.",
    "",
    "Excluded objects:",
]

for _, row in excluded.iterrows():
    methods.append(
        f"Image {int(row['IMAGE_NUMBER']):03d}; DES {row['DES_ID']}: "
        f"{row['REVIEW_NOTE']}"
    )

methods += [
    "",
    "Column definitions:",
    "IMAGE_NUMBER: original global thumbnail number; gaps are intentional.",
    "DES_ID: DES DR2 coadd_object_id.",
    "RA_DEG / DEC_DEG: copied unchanged from the working catalogue's "
    "GAIA_RA / GAIA_DEC columns. Their recorded source is DES DR2 "
    "ALPHAWIN_J2000 / DELTAWIN_J2000; they are not newly measured Gaia positions.",
    "DES_R_MAG / DES_R_MAG_ERR: DES DR2 MAG_AUTO_R and its reported "
    "measurement uncertainty; observed AB magnitudes, not dereddened.",
    "Quoted magnitude errors do not encompass all blending or other "
    "systematic photometry errors.",
    "REVIEW_STATUS: current inclusion/review decision, not a definitive "
    "physical classification.",
    "REVIEW_NOTE: selected visual and catalogue concerns for supervisor review.",
    "",
    "Status meanings:",
    "RETAIN: included in this provisional list.",
    "RETAIN_WITH_CAVEAT: included despite a recorded uncertainty or image issue.",
    "FOLLOW_UP: included, but specifically requires further investigation.",
    "",
    "Evidence:",
    "Visual review: GW190814_273_review_sheets.pdf.",
    "Catalogue follow-up: GW190814_15_object_catalogue_checks_v2.zip.",
    "Detailed catalogue follow-up covered 15 selected objects, not all 273.",
    "Gaia classifier scores are model outputs, not verified identities.",
    "Legacy point-source treatment for the four excluded objects used Gaia "
    "information and is not fully independent confirmation.",
    "No usable Gaia parallax/proper motion was retrieved for those four.",
    "No SDSS spectra were found within 3 arcsec for the 15 checked objects; "
    "this was not an exhaustive search of existing spectroscopy.",
    "",
    "Limitations:",
    "This compact table omits most raw extraction/mask flags. RETAIN or a "
    "blank note does not mean every catalogue quality flag is zero.",
    "Retain the original 273-target CSV and v2 evidence ZIP as the detailed audit.",
    "Images 098, 123 and 266 remain marked FOLLOW_UP.",
    "The photometry discrepancy for image 266 has not been corrected.",
    "This is a supervisor-review CSV, not a validated WASP FITS catalogue.",
    "Observing priorities and final observing configuration are not assigned here.",
]

methods_path.write_text("\n".join(methods) + "\n", encoding="utf-8")


# ============================================================
# RESULTS AND DOWNLOAD
# ============================================================

print(f"Original sample: {len(df)}")
print(f"Excluded probable stars: {len(excluded)}")
print(f"Retained for Mark: {len(retained)}")
print(f"Columns in Mark's CSV: {len(retained.columns)}")

print("\nExcluded thumbnail numbers:")
print(excluded["IMAGE_NUMBER"].tolist())

print("\nReview status counts:")
print(retained["REVIEW_STATUS"].value_counts().to_string())

print("\nObjects specifically requiring follow-up:")
display(retained.loc[retained["REVIEW_STATUS"].eq("FOLLOW_UP")])

print("\nFiles created:")
for path in [main_path, excluded_path, methods_path]:
    display(FileLink(str(path)))

# In Colab, download the main CSV automatically.
# The exclusion log and methods note remain in the output folder.
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(main_path))

Original sample: 273
Excluded probable stars: 4
Retained for Mark: 269
Columns in Mark's CSV: 8

Excluded thumbnail numbers:
[12, 30, 173, 185]

Review status counts:
REVIEW_STATUS
RETAIN                252
RETAIN_WITH_CAVEAT     14
FOLLOW_UP               3

Objects specifically requiring follow-up:


,IMAGE_NUMBER,DES_ID,RA_DEG,DEC_DEG,DES_R_MAG,DES_R_MAG_ERR,REVIEW_STATUS,REVIEW_NOTE
97,98,1113986212,13.3522825611079,-25.3864302783852,19.581577,0.0042148554,FOLLOW_UP,Gaia quasar score 0.9989; Legacy extended mode...
122,123,1095342910,10.7876083294035,-24.673209932959,17.746683,0.0010559816,FOLLOW_UP,Conflicting classification: Gaia star score 0....
265,266,1124855760,14.2728404413631,-25.2068499890215,20.78097,0.054932628,FOLLOW_UP,Legacy extended model supports retention. Phot...



Files created:


/content/GW190814_Mark_review/GW190814_269_targets_for_Mark.csv

/content/GW190814_Mark_review/GW190814_four_excluded_targets.csv

/content/GW190814_Mark_review/GW190814_selection_notes.txt

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>